# 05 — Figures and tables

Loads the saved arrays only (run notebooks 02–04 first) and writes the report figures to `results/figures/` (300 dpi) and the results table to `results/table.csv`. All figures plot **error rate = 1 − accuracy**, mean over splits, with ± 1 std where shown.

The sample image grid (`samples.png`) is made in notebook 01.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

from src.config import RESULTS_DIR

ARRAYS = RESULTS_DIR / "arrays"
FIGURES = RESULTS_DIR / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", "Mean of empty slice", RuntimeWarning)          # min-Mahal is nan above the cap
warnings.filterwarnings("ignore", "Degrees of freedom <= 0", RuntimeWarning)

# Colour = classifier identity, fixed across every figure; marker is the second cue (print / colour-blind)
STYLE = {
    "1-NN":       dict(color="#2a78d6", marker="o"),
    "min-Euclid": dict(color="#eb6834", marker="s"),
    "min-Mahal":  dict(color="#1baf7a", marker="^"),
}
EPS_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]   # ordinal: small -> large eps
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "axes.labelcolor": INK2, "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "grid.linestyle": "-",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 1.6, "lines.markersize": 5, "legend.frameon": False, "legend.fontsize": 8,
    "legend.labelcolor": INK, "savefig.dpi": 300, "savefig.bbox": "tight",
})


def pct(ax):
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))


def save(fig, name):
    fig.savefig(FIGURES / name)
    plt.show()

In [ ]:
pca = np.load(ARRAYS / "pca_acc.npz")
base = np.load(ARRAYS / "baseline.npz")
eig = np.load(ARRAYS / "eigvals.npy")
lda = np.load(ARRAYS / "lda_acc.npz")
lda_m = np.load(ARRAYS / "lda_m_sweep.npz")
sel = np.load(ARRAYS / "selected.npz")
e4 = np.load(ARRAYS / "trainsize_acc.npz")
e5 = np.load(ARRAYS / "random_split_acc.npz")
e6 = np.load(ARRAYS / "reg_mahal_acc.npz")
e6_sel = np.load(ARRAYS / "reg_mahal_selected.npz")
names = [str(n) for n in pca["classifiers"]]
d_list = pca["d_list"]
mark_at = [np.flatnonzero(d_list == d)[0] for d in (1, 2, 5, 10, 20, 50, 100, 200)]

## Figure: eigenspectrum (E3)

In [ ]:
k = np.arange(1, eig.shape[1] + 1)
cum = np.cumsum(eig, axis=1) / eig.sum(axis=1, keepdims=True)
blue = STYLE["1-NN"]["color"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 2.8))
ax1.fill_between(k, eig.min(axis=0), eig.max(axis=0), color=blue, alpha=0.2, lw=0)
ax1.semilogy(k, eig.mean(axis=0), color=blue)
ax1.set(xlabel="component index", ylabel="eigenvalue", title="(a) PCA eigenvalues", xlim=(0, k[-1] + 1))
ax2.plot(k, cum.mean(axis=0), color=blue)
for p, y_text in ((0.90, 0.60), (0.95, 0.35), (0.99, 0.60)):   # staggered: 43 and 72 are close
    d_p = int(np.median((cum < p).sum(axis=1) + 1))
    ax2.plot([d_p, d_p], [0, p], color=AXIS, lw=0.8, zorder=0)
    ax2.plot(d_p, p, "o", color=blue, mec="white", mew=1.5, ms=7)
    ax2.annotate(f"{p:.0%}\nd = {d_p}", (d_p, y_text), ha="center", va="center", color=INK, fontsize=8,
                 bbox=dict(fc="white", ec="none", pad=1.5))
ax2.set(xlabel="number of components d", ylabel="cumulative variance", title="(b) Variance retained",
        ylim=(0, 1.02), xlim=(0, k[-1] + 1))
pct(ax2)
fig.tight_layout()
save(fig, "eigenspectrum.png")

## Figure: PCA error rate vs d (E0, E1)

Shaded band = ± 1 std over the 6 splits. Open markers = validation-selected d (median over splits). Dashed lines = raw-pixel baselines.

In [ ]:
err, sd = 1 - np.nanmean(pca["acc"], axis=0), np.nanstd(pca["acc"], axis=0)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for i, n in enumerate(names):
    s = STYLE[n]
    ok = ~np.isnan(err[i])
    ax.fill_between(d_list[ok], (err[i] - sd[i])[ok], (err[i] + sd[i])[ok], color=s["color"], alpha=0.15, lw=0)
    ax.plot(d_list[ok], err[i][ok], color=s["color"], marker=s["marker"],
            markevery=[m for m in mark_at if ok[m]], label=n)
    j = np.argmin(np.abs(d_list - np.median(sel["pca_d"][:, i])))
    ax.plot(d_list[j], err[i, j], s["marker"], ms=9, mfc="white", mec=s["color"], mew=1.8, zorder=5)
for i, n in enumerate(names[:2]):
    ax.axhline(1 - base["acc"][:, i].mean(), color=STYLE[n]["color"], lw=1, ls="--", label=f"{n}, raw pixels (n = 1024)")
ax.axvline(220, color=MUTED, lw=0.8, zorder=0)
ax.annotate("N − c = 220\n(min-Mahal cap)", (220, 0.27), xytext=(-6, 0), textcoords="offset points",
            ha="right", color=INK2, fontsize=8)
ax.plot([], [], "o", ms=8, mfc="white", mec=INK2, mew=1.5, label="validation-selected d")
ax.set(xscale="log", xlabel="PCA dimension d (log scale)", ylabel="test error rate",
       title="PCA: error rate vs dimension", ylim=(0, 0.7))
pct(ax)
ax.legend(ncol=2, loc="upper right")
fig.tight_layout()
save(fig, "e1_pca_error_vs_d.png")

## Figure: PCA → LDA (E2)

In LDA space min-Mahal and min-Euclid give identical predictions ($W^T S_w W = I$), so they are drawn as one line.

In [ ]:
dl = lda["d_list"]
err_l, sd_l = 1 - lda["acc"].mean(axis=0), lda["acc"].std(axis=0)
m_list = lda_m["m_list"]
acc_m = lda_m["acc"][:, :, :, -1]   # d = c - 1 = 19
err_m, sd_m = 1 - acc_m.mean(axis=1), acc_m.std(axis=1)
shown = {"1-NN": "1-NN", "min-Euclid": "min-Euclid = min-Mahal"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.0))
# the two curves almost coincide at m = N - c: one band, 1-NN drawn on top with offset markers
ax1.fill_between(dl, err_l[1] - sd_l[1], err_l[1] + sd_l[1], color=STYLE["min-Euclid"]["color"], alpha=0.15, lw=0)
for i, n in reversed(list(enumerate(names[:2]))):
    s = STYLE[n]
    ax1.plot(dl, err_l[i], color=s["color"], marker=s["marker"], lw=1.6 if i else 1.2,
             markevery=(0, 3) if i else (1, 3), label=shown[n])
for i, n in enumerate(names[:2]):
    s = STYLE[n]
    ax2.errorbar(m_list, err_m[:, i], yerr=sd_m[:, i], color=s["color"], marker=s["marker"],
                 capsize=2, elinewidth=0.8, label=shown[n])
ax1.set(xlabel="LDA dimension d", ylabel="test error rate", title="(a) m = N − c = 220",
        xticks=[1, 4, 7, 10, 13, 16, 19], ylim=(0, 0.9))
ax2.set(xlabel="intermediate PCA dimension m", ylabel="test error rate at d = 19", title="(b) Effect of m", ylim=(0, 0.9))
for ax in (ax1, ax2):
    pct(ax)
ax1.annotate("band: ± 1 std over splits", (19, 0.62), ha="right", color=INK2, fontsize=8)
h, l = ax1.get_legend_handles_labels()
ax1.legend(h[::-1], l[::-1], loc="upper right")
fig.tight_layout()
save(fig, "e2_lda_error.png")

## Figure: training size (E4)

Test error at validation-selected d (PCA) and (m, d) (PCA → LDA). Error bars = ± 1 std over splits. Solid/filled = PCA, dashed/open = PCA → LDA.

In [ ]:
steps = e4["steps"]
per_class = 72 // steps
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for i, n in enumerate(names):
    s = STYLE[n]
    for method, ls in (("pca", "-"), ("lda", "--")):
        if method == "lda" and n == "min-Mahal":
            continue   # identical to PCA -> LDA min-Euclid
        acc = [e4[f"{method}_test_{st}"][:, i] for st in steps]
        label = f"PCA {n}" if method == "pca" else ("PCA→LDA min-Euclid = min-Mahal" if n == "min-Euclid" else f"PCA→LDA {n}")
        ax.errorbar(per_class, [1 - a.mean() for a in acc], yerr=[a.std() for a in acc], color=s["color"],
                    marker=s["marker"], ls=ls, mfc="white" if method == "lda" else s["color"],
                    capsize=2, elinewidth=0.8, label=label)
ax.set(xlabel="training images per class", ylabel="test error rate", title="Error rate vs training size",
       xticks=per_class, ylim=(0, None))
pct(ax)
ax.legend(ncol=2)
fig.tight_layout()
save(fig, "e4_trainsize.png")

## Figure: random vs pose-spaced split (E5)

Solid/filled = pose-spaced, dashed/open = random (same training size, 6 seeds).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
for ax, n_pc in zip(axes, (12, 36)):
    for kind, ls in (("spaced", "-"), ("random", "--")):
        e = 1 - np.nanmean(e5[f"{kind}_{n_pc}"], axis=0)
        for i, n in enumerate(names):
            ok = ~np.isnan(e[i])
            s = STYLE[n]
            ax.plot(e5["d_list"][ok], e[i][ok], color=s["color"], ls=ls, marker=s["marker"],
                    mfc="white" if kind == "random" else s["color"],
                    markevery=[m for m in mark_at if ok[m]], label=f"{n}, {kind}")
    ax.set(xscale="log", xlabel="PCA dimension d (log scale)", title=f"{n_pc} training images per class", ylim=(0, 0.5))
    pct(ax)
axes[0].set_ylabel("test error rate")
axes[1].legend(fontsize=7)
fig.tight_layout()
save(fig, "e5_random_vs_spaced.png")

## Figure: regularised Mahalanobis (E6)

In [ ]:
eps_list = e6["eps_list"]
err6 = 1 - np.nanmean(e6["acc"], axis=0)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for i, (e, c) in enumerate(zip(eps_list, EPS_RAMP)):
    ok = ~np.isnan(err6[i])
    ax.plot(d_list[ok], err6[i][ok], color=c, label=f"ε = {e:g}" + (" (unregularised)" if e == 0 else ""))
ax.axvline(220, color=MUTED, lw=0.8, zorder=0)
ax.annotate("N − c = 220", (220, 0.45), xytext=(-6, 0), textcoords="offset points", ha="right", color=INK2, fontsize=8)
ax.set(xscale="log", xlabel="PCA dimension d (log scale)", ylabel="test error rate",
       title=r"Regularised min-Mahal: $\Sigma_w + \epsilon\,(\mathrm{tr}\,\Sigma_w / d)\,I$",
       xlim=(8, 260), ylim=(0, 0.5))   # all eps coincide below d ~ 50
pct(ax)
ax.legend(loc="upper center")
fig.tight_layout()
save(fig, "e6_reg_mahal.png")

## Results table

Test accuracy (mean ± std over splits) at the validation-selected value, plus the best point on the test curve for reference (**oracle**: chosen with test data, not a valid result).

In [ ]:
def spread(a):
    a = np.asarray(a)
    return f"{a.min():g}" if a.min() == a.max() else f"{np.median(a):g} [{a.min():g}–{a.max():g}]"

rows = []
for i, n in enumerate(names[:2]):
    a = base["acc"][:, i]
    rows.append(("Raw pixels", n, "n = 1024", a.mean(), a.std(), np.nan))
best_pca = np.nanmax(np.nanmean(pca["acc"], axis=0), axis=1)
for i, n in enumerate(names):
    a = sel["pca_test"][:, i]
    rows.append(("PCA", n, f"d = {spread(sel['pca_d'][:, i])}", a.mean(), a.std(), best_pca[i]))
for i, n in enumerate(names):
    a = lda["acc"][:, i, -1]
    rows.append(("PCA(N − c) → LDA", n, "fixed: m = 220, d = 19", a.mean(), a.std(), lda["acc"][:, i].mean(axis=0).max()))
best_lda = lda_m["acc"].mean(axis=1).max(axis=(0, 2))
for i, n in enumerate(names):
    a = sel["lda_test"][:, i]
    rows.append(("PCA(m) → LDA", n, f"m = {spread(sel['lda_m'][:, i])}, d = {spread(sel['lda_d'][:, i])}",
                 a.mean(), a.std(), best_lda[i]))
a = e6_sel["test"]
eps_chosen = ", ".join(f"{e:g}" for e in sorted(set(e6_sel["eps"])))
rows.append(("PCA, regularised", "min-Mahal", f"ε ∈ {{{eps_chosen}}}, d = {spread(e6_sel['d'])}",
             a.mean(), a.std(), np.nanmax(np.nanmean(e6["acc"], axis=0))))

table = pd.DataFrame(rows, columns=["method", "classifier", "selected by validation",
                                    "test acc mean", "test acc std", "best on test curve (oracle)"])
table.to_csv(RESULTS_DIR / "table.csv", index=False, float_format="%.4f")
view = table[["method", "classifier", "selected by validation"]].copy()
view["test accuracy"] = [f"{m:.1%} ± {s:.1%}" for m, s in zip(table["test acc mean"], table["test acc std"])]
view["oracle"] = ["–" if np.isnan(b) else f"{b:.1%}" for b in table["best on test curve (oracle)"]]
view

In [ ]:
sorted(p.name for p in FIGURES.glob("*.png"))